# Otimização — capítulo 4: Otimização com restrições
### Exemplos em Python

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aguilarmadeira/optimization-course/blob/main/code/python/notebooks/04_constrained_optimization.ipynb)

Cada secção corre o exemplo `ex….py` do método e compara os resultados com os slides (no fim: *confere com os slides: sim*). Comece pela célula **Preparação**; no Colab, *Runtime → Run all* corre tudo.

Os slides usam vírgula decimal; aqui usa-se o ponto. Este caderno é gerado a partir dos ficheiros `ex….py` por `make_notebooks.py`.

*Examples for Chapter 4 of the course "Optimization" (ISEL, J. F. A. Madeira). Code comments and printed messages are in Portuguese; the English slides are in [`notes/en/`](https://github.com/aguilarmadeira/optimization-course/tree/main/notes/en).*

## Preparação

In [1]:
# Preparação: põe as funções da UC no caminho do Python.
# No Colab (ou noutra pasta), clona o repositório; dentro do repositório, usa code/python.
import os, sys
LOCAL = [p for p in ("..", os.path.join("..", "..")) if os.path.isfile(os.path.join(p, "uc_setup.py"))]
if LOCAL:
    RAIZ = os.path.abspath(LOCAL[0])
else:
    if not os.path.isdir("optimization-course"):
        !git clone -q --depth 1 https://github.com/aguilarmadeira/optimization-course
    RAIZ = os.path.abspath(os.path.join("optimization-course", "code", "python"))
if RAIZ not in sys.path:
    sys.path.insert(0, RAIZ)
import uc_setup  # acrescenta as pastas dos métodos e common/
print("Código da UC pronto.")

Código da UC pronto.


## 4.1 Extremos condicionados: multiplicadores de Lagrange

Exemplo [`ex04_1_lagrange.py`](https://github.com/aguilarmadeira/optimization-course/blob/main/code/python/04_constrained_optimization/04_1_lagrange/ex04_1_lagrange.py)

```text
Reproduz os exemplos do deck 4.1 (multiplicadores de Lagrange).

(a) Resolve os exemplos dos slides pelas condições de Lagrange, com as contas
    do slide, e verifica numericamente as condições
    grad f(x*) + sum_l lambda_l grad h_l(x*) = 0  e  h_l(x*) = 0.
    Convenção das aulas:  L(x, lambda) = f(x) + sum_l lambda_l h_l(x).
    (Com L = f - sum lambda_l h_l, como em Nocedal e Wright, só muda o sinal de lambda.)
      1. O exemplo resolvido: min 2x1^2 + x2^2 s.a. x1 + x2 = 1
         (e «Geometria (1)» e «O que significa lambda?»).
      2. Candidatos: min x1 + x2 s.a. x1^2 + x2^2 = 2, classificados também
         com a Hessiana orlada (n = 2, uma restrição: det < 0 mínimo, det > 0 máximo).
      3. Exemplo 2, a lata: min 2 pi r^2 + 2 pi r h s.a. pi r^2 h = V.
      4. Várias restrições: min x1^2+x2^2+x3^2 s.a. x1+x2+x3 = 3, x1-x2 = 1.
      5. Quando a condição de Lagrange falha (gradientes dependentes).
(b) Resolve 1, 3 e 4 com scipy.optimize.minimize(method='trust-constr'),
    como no deck 4.2 («No computador»); os multiplicadores vêm em r.v.
    Sinal: o SciPy escreve L = f + v'c, com c = h e limites 0 <= c <= 0, logo
    lambda = r.v, sem troca de sinal (como no fmincon: lambda = lambda.eqnonlin).

Os slides usam vírgula decimal; aqui usa-se o ponto.
Não há função da UC neste deck: é um exemplo. Implementação didática.
```

In [2]:
import warnings

import numpy as np
from scipy.optimize import NonlinearConstraint, minimize

# Com restrições lineares, o quasi-Newton do trust-constr avisa que a Hessiana
# da restrição é nula («delta_grad == 0.0»); é esperado e não afeta o resultado.
warnings.filterwarnings("ignore", message="delta_grad == 0.0")


def confere(v, s, d):
    """v confere com o valor s do slide, mostrado com d casas decimais?"""
    v = np.ravel(np.asarray(v, float)); s = np.ravel(np.asarray(s, float))
    d = np.ravel(np.asarray(d, float))
    return bool(np.all(np.abs(v - s) <= 0.5 * 10.0 ** (-d) * (1 + 1e-6)))


def perto(v, s, tol):
    """v está a menos de tol do valor exato s?"""
    return bool(np.all(np.abs(np.ravel(np.asarray(v, float))
                              - np.ravel(np.asarray(s, float))) <= tol))


def simnao(c):
    return tuple("sim" if ci else "não" for ci in c)


def orlada(gh, HL):
    """Hessiana orlada (n = 2, uma restrição): [[0, gh'], [gh, HL]]."""
    gh = np.ravel(gh)
    return np.block([[np.zeros((1, 1)), gh[None, :]], [gh[:, None], np.atleast_2d(HL)]])


def lagrange_lin(Q, c, A, b):
    """min 1/2 x'Qx + c'x s.a. Ax = b: grad_x L = Qx + c + A'lambda = 0, Ax = b."""
    Q = np.atleast_2d(Q); A = np.atleast_2d(A); m = A.shape[0]
    M = np.block([[Q, A.T], [A, np.zeros((m, m))]])
    z = np.linalg.solve(M, np.concatenate([-np.ravel(c), np.ravel(b)]))
    return z[:Q.shape[0]], z[Q.shape[0]:]


fmt = lambda v: ", ".join("%.4f" % t for t in np.ravel(v))
ok = True
tolA = 1e-10        # contas exatas (a)
tolB = 1e-5         # solver (b)

print("Otimização — deck 4.1: multiplicadores de Lagrange")
print("Convenção das aulas: L = f + sum lambda_l h_l")
print("\n=============== (a) pelas condições de Lagrange ===============")

# ------------------------------------------------ 1. O exemplo resolvido
f1 = lambda x: 2 * x[0]**2 + x[1]**2
df1 = lambda x: np.array([4 * x[0], 2 * x[1]])
h1 = lambda x: x[0] + x[1] - 1
dh1 = np.array([1.0, 1.0])
print("\n1. min 2x1^2 + x2^2  s.a.  h = x1 + x2 - 1 = 0")
xg = np.array([0.8, 0.2]); d = np.array([1.0, -1.0])
print("   Geometria (1): em x = (0.8, 0.2), grad f = (%.1f, %.1f), grad f'd = %.1f (d = (1,-1))"
      % (*df1(xg), df1(xg) @ d))
c = [confere(df1(xg), [3.2, 0.4], 1), confere(df1(xg) @ d, 2.8, 1)]
# 4x1 + lambda = 0, 2x2 + lambda = 0, x1 + x2 = 1
x, lam = lagrange_lin(np.diag([4.0, 2.0]), [0, 0], [[1, 1]], [1])
res = np.linalg.norm(df1(x) + lam[0] * dh1, np.inf)
print("   Sistema: 4x1 + lambda = 0, 2x2 + lambda = 0, x1 + x2 = 1")
print("   x* = (%.4f, %.4f), lambda* = %.4f, f* = %.4f  (slide: (1/3, 2/3), -4/3, 2/3)"
      % (*x, lam[0], f1(x)))
print("   grad f(x*) = (%.4f, %.4f) = -lambda*(1,1);  |grad_x L| = %.1e, |h| = %.1e"
      % (*df1(x), res, abs(h1(x))))
detH1 = np.linalg.det(orlada(dh1, np.diag([4.0, 2.0])))
print("   Hessiana orlada [[0,1,1],[1,4,0],[1,0,2]]: det = %.4f < 0 -> mínimo" % detH1)
# O que significa lambda?  x1 + x2 = 1 + delta  ->  df*/d(delta) = -lambda*
fd = lambda delta: f1(lagrange_lin(np.diag([4.0, 2.0]), [0, 0], [[1, 1]], [1 + delta])[0])
dl = 1e-4
decl = (fd(dl) - fd(-dl)) / (2 * dl)
print("   Sensibilidade: f*(delta) = (2/3)(1+delta)^2, declive em 0 (dif. centrais) = %.6f = -lambda*"
      % decl)
c += [perto(x, [1/3, 2/3], tolA), perto(lam, -4/3, tolA), perto(f1(x), 2/3, tolA),
      res <= tolA, abs(h1(x)) <= tolA, perto(decl, 4/3, 1e-8), perto(detH1, -6, 1e-9)]
print("   Geometria (1): %s %s | x*: %s | lambda*: %s | f*: %s | grad_x L = 0: %s | h = 0: %s"
      " | declive = -lambda*: %s | orlada -6: %s" % simnao(c))
ok = ok and all(c)

# ------------------------------------------------ 2. Candidatos
f2 = lambda x: x[0] + x[1]
df2 = lambda x: np.array([1.0, 1.0])
h2 = lambda x: x[0]**2 + x[1]**2 - 2
dh2 = lambda x: np.array([2 * x[0], 2 * x[1]])
print("\n2. min x1 + x2  s.a.  h = x1^2 + x2^2 - 2 = 0")
print("   (1,1) + lambda (2x1, 2x2) = 0  =>  x1 = x2 = -1/(2 lambda);  na restrição, x1 = x2 = +-1")
dT = np.array([1.0, -1.0]) / np.sqrt(2)          # direção tangente (a mesma nos dois)
c = []
for x, slam, sf, sdet, nome in [((-1.0, -1.0), 0.5, -2, -8, "mínimo"), ((1.0, 1.0), -0.5, 2, 8, "máximo")]:
    x = np.array(x)
    lam = -1 / (2 * x[0])
    res = np.linalg.norm(df2(x) + lam * dh2(x), np.inf)
    H = 2 * lam * np.eye(2)                      # Hessiana de L em x
    q = dT @ H @ dT                              # 2.ª ordem em T
    dH = np.linalg.det(orlada(dh2(x), H))        # Hessiana orlada: det = -d'H d, d = (h_x2, -h_x1)
    print("   x = (%2d, %2d): lambda = %5.2f, f = %2d, |grad_x L| = %.1e, |h| = %.1e, "
          "d'H_L d = %5.2f, det orlada = %5.2f -> %s" % (*x, lam, f2(x), res, abs(h2(x)), q, dH, nome))
    c += [perto(lam, slam, tolA), perto(f2(x), sf, tolA), res <= tolA, abs(h2(x)) <= tolA,
          np.sign(q) == np.sign(lam), perto(dH, sdet, 1e-9), np.sign(dH) == -np.sign(q)]
print("   lambda, f, grad_x L = 0, h = 0 e classificação (2.ª ordem e Hessiana orlada) nos dois candidatos: %s"
      % simnao([all(c)]))
ok = ok and all(c)

# ------------------------------------------------ 3. A lata
V = 330.0
A3 = lambda z: 2 * np.pi * z[0]**2 + 2 * np.pi * z[0] * z[1]      # z = (r, h), h = altura
dA3 = lambda z: np.array([4 * np.pi * z[0] + 2 * np.pi * z[1], 2 * np.pi * z[0]])
h3 = lambda z: np.pi * z[0]**2 * z[1] - V
dh3 = lambda z: np.array([2 * np.pi * z[0] * z[1], np.pi * z[0]**2])
print("\n3. Lata: min A = 2 pi r^2 + 2 pi r h  s.a.  pi r^2 h = V = %g cm^3" % V)
print("   dL/dh = 2 pi r + lambda pi r^2 = 0 => lambda = -2/r;  dL/dr = 0 => h = 2r;  pi r^2 (2r) = V")
r = (V / (2 * np.pi))**(1 / 3); h = 2 * r; lam = -2 / r; z = np.array([r, h])
res = np.linalg.norm(dA3(z) + lam * dh3(z), np.inf)
HL = np.pi * np.array([[4 + 2 * lam * h, 2 + 2 * lam * r],
                       [2 + 2 * lam * r, 0.0]])  # Hessiana de L = pi [-4 -2; -2 0]
dT = np.array([1.0, -4.0])                       # dh3(z)'dT = 0
print("   r* = %.4f cm, h* = %.4f cm, A* = %.4f cm^2, lambda* = -2/r* = %.4f cm^2/cm^3 (dA*/dV = -lambda*)"
      % (r, h, A3(z), lam))
print("   |grad_x L| = %.1e, |h| = %.1e;  2.ª ordem: d = (1,-4) em T, d'H_L d = %.4f = 12 pi > 0 (mínimo)"
      % (res, abs(h3(z)), dT @ HL @ dT))
c = [confere(r, 3.745, 3), confere(r, 3.74, 2), confere(h, 7.49, 2), confere(A3(z), 264.4, 1),
     confere(lam, -0.534, 3), res <= 1e-9 * np.linalg.norm(dA3(z)), abs(h3(z)) <= 1e-9 * V,
     perto(dT @ HL @ dT, 12 * np.pi, 1e-9)]
print("   r* = 3.745 (3.74): %s %s | h* = 7.49: %s | A* = 264.4: %s | lambda* = -0.534: %s"
      " | grad_x L = 0: %s | h = 0: %s | 12 pi: %s" % simnao(c))
ok = ok and all(c)

# ------------------------------------------------ 4. Várias restrições
f4 = lambda x: x[0]**2 + x[1]**2 + x[2]**2
df4 = lambda x: 2 * np.asarray(x)
A4 = np.array([[1.0, 1, 1], [1, -1, 0]]); b4 = np.array([3.0, 1])   # h = A4 x - b4
print("\n4. min x1^2 + x2^2 + x3^2  s.a.  x1 + x2 + x3 = 3,  x1 - x2 = 1")
x, lam = lagrange_lin(2 * np.eye(3), np.zeros(3), A4, b4)
res = np.linalg.norm(df4(x) + A4.T @ lam, np.inf)
d = np.array([1.0, 1, -2])
print("   x* = (%s), lambda* = (%s), f* = %.4f  (slide: (3/2, 1/2, 1), (-2, -1), 7/2)"
      % (fmt(x), fmt(lam), f4(x)))
print("   grad f(x*) = (%g, %g, %g) = 2 (1,1,1) + 1 (1,-1,0) = -lambda1 grad h1 - lambda2 grad h2;"
      "  grad f'd = %g (d = (1,1,-2));"
      "  |grad_x L| = %.1e, |h| = %.1e"
      % (*df4(x), df4(x) @ d + 0.0, res, np.linalg.norm(A4 @ x - b4, np.inf)))
c = [perto(x, [1.5, 0.5, 1], tolA), perto(lam, [-2, -1], tolA), perto(f4(x), 3.5, tolA),
     perto(df4(x) @ d, 0, tolA), res <= tolA, np.linalg.norm(A4 @ x - b4, np.inf) <= tolA]
print("   x*: %s | lambda*: %s | f*: %s | grad f'd = 0: %s | grad_x L = 0: %s | h = 0: %s" % simnao(c))
ok = ok and all(c)

# ------------------------------------------------ 5. Quando Lagrange falha
print("\n5. min x1 + x2 + x3^2  s.a.  x1 - 1 = 0,  x1^2 + x2^2 - 1 = 0;  x* = (1, 0, 0)")
x = np.array([1.0, 0, 0])
gf = np.array([1.0, 1, 2 * x[2]])
J = np.array([[1.0, 0, 0], [2 * x[0], 2 * x[1], 0]])   # linhas: grad h1', grad h2'
lam = np.linalg.lstsq(J.T, -gf, rcond=None)[0]         # melhor lambda (mínimos quadrados)
resmin = np.linalg.norm(gf + J.T @ lam)
print("   grad h1 = (%g,%g,%g), grad h2 = (%g,%g,%g): característica %d (dependentes)"
      % (*J[0], *J[1], np.linalg.matrix_rank(J)))
print("   min_lambda |grad f + J'lambda| = %.4f: dL/dx2 = 1 para todos os lambda -> sistema sem solução"
      % resmin)
c = [np.linalg.matrix_rank(J) == 1, perto(resmin, 1, tolA)]
print("   gradientes dependentes: %s | sem multiplicadores (resíduo 1): %s" % simnao(c))
ok = ok and all(c)

# =================================================== (b) com um solver
print("\n=============== (b) com um solver ===============")
print("Solver: scipy.optimize.minimize(method='trust-constr'), multiplicadores em r.v.")
print("O SciPy escreve L = f + v'c (aqui c = h, com 0 <= c <= 0), como nas aulas: lambda = r.v.")
opts = dict(gtol=1e-10, xtol=1e-12, maxiter=2000)
probs = [(f1, df1, lambda x: [h1(x)], lambda x: [dh1], [0, 0], [1/3, 2/3], [-4/3],
          "1. exemplo resolvido"),
         (A3, dA3, lambda z: [h3(z)], lambda z: [dh3(z)], [3, 6], None, None,
          "3. lata (x0 = (3, 6))"),
         (f4, df4, lambda x: A4 @ x - b4, lambda x: A4, [0, 0, 0], [1.5, 0.5, 1], [-2, -1],
          "4. várias restrições")]
for fo, dfo, hc, Jh, x0, xs, ls, nome in probs:
    r = minimize(fo, x0, jac=dfo, method="trust-constr", options=opts,
                 constraints=[NonlinearConstraint(hc, 0, 0, jac=Jh)])
    lam = r.v[0]
    print("   %s: x = (%s), f = %.4f, lambda = r.v = (%s)  [status %d, %d it., %d aval. de f]"
          % (nome, fmt(r.x), r.fun, fmt(lam), r.status, r.nit, r.nfev))
    if xs is None:      # a lata: comparar com os valores do slide
        c = [confere(r.x[0], 3.745, 3), confere(r.x[1], 7.49, 2), confere(r.fun, 264.4, 1),
             confere(lam, -0.534, 3)]
        print("      r*: %s | h*: %s | A*: %s | lambda*: %s" % simnao(c))
    else:
        c = [perto(r.x, xs, tolB), perto(lam, ls, tolB)]
        print("      x*: %s | lambda*: %s" % simnao(c))
    ok = ok and all(c)

print("\nconfere com os slides: %s" % ("sim" if ok else "não"))


Otimização — deck 4.1: multiplicadores de Lagrange
Convenção das aulas: L = f + sum lambda_l h_l

=============== (a) pelas condições de Lagrange ===============

1. min 2x1^2 + x2^2  s.a.  h = x1 + x2 - 1 = 0
   Geometria (1): em x = (0.8, 0.2), grad f = (3.2, 0.4), grad f'd = 2.8 (d = (1,-1))
   Sistema: 4x1 + lambda = 0, 2x2 + lambda = 0, x1 + x2 = 1
   x* = (0.3333, 0.6667), lambda* = -1.3333, f* = 0.6667  (slide: (1/3, 2/3), -4/3, 2/3)
   grad f(x*) = (1.3333, 1.3333) = -lambda*(1,1);  |grad_x L| = 0.0e+00, |h| = 0.0e+00
   Hessiana orlada [[0,1,1],[1,4,0],[1,0,2]]: det = -6.0000 < 0 -> mínimo
   Sensibilidade: f*(delta) = (2/3)(1+delta)^2, declive em 0 (dif. centrais) = 1.333333 = -lambda*
   Geometria (1): sim sim | x*: sim | lambda*: sim | f*: sim | grad_x L = 0: sim | h = 0: sim | declive = -lambda*: sim | orlada -6: sim

2. min x1 + x2  s.a.  h = x1^2 + x2^2 - 2 = 0
   (1,1) + lambda (2x1, 2x2) = 0  =>  x1 = x2 = -1/(2 lambda);  na restrição, x1 = x2 = +-1
   x = (-1, -1): la

## 4.2 Condições de Karush–Kuhn–Tucker (KKT)

Exemplo [`ex04_2_kkt.py`](https://github.com/aguilarmadeira/optimization-course/blob/main/code/python/04_constrained_optimization/04_2_kkt/ex04_2_kkt.py) · funções da UC: [`kkt_check.py`](https://github.com/aguilarmadeira/optimization-course/blob/main/code/python/04_constrained_optimization/04_2_kkt/kkt_check.py)

```text
Reproduz os exemplos do deck 4.2 (condições KKT).

(a) Resolve os exemplos dos slides pelas condições KKT, com as contas do
    slide, e verifica-as numericamente com kkt_check (estacionariedade,
    admissibilidade, complementaridade u_j g_j = 0 e sinal u_j >= 0).
    Convenção da UC: g_j(x) >= 0, h_l(x) = 0, L = f - u'g - lambda'h.
    Nas aulas: L = f - lambda'g - beta'h; no código, u = lambda_j e lambda = beta_l das aulas.
      1. De 4.1 para 4.2 (círculo largo / apertado) e Geometria (2) (u < 0).
      2. Exemplo 1: os 2^J = 4 casos da receita; sobrevive o caso 3.
      3. «Cuidado: ativa não implica u > 0»: min x^2 s.a. x >= 0.
      4. Exemplo 2: verificar o candidato (2,1): u1 = 1/3, u2 = 2/3.
      5. «KKT dá candidatos»: min -x^2 s.a. x + 1 >= 0, 2 - x >= 0.
      6. O que medem os multiplicadores: exemplo 1 com b = 3.1 e a produção
         do 1.1 (preços-sombra 10, 20, 0).
      7. Quando KKT falha: LICQ.
      8. Exercícios 1, 2 e 4 (as respostas a cinzento).
(b) Resolve o exemplo 1, o exemplo 2 e os exercícios 2 e 4 com
    scipy.optimize.minimize(method='trust-constr'), como no slide
    «No computador»; os multiplicadores vêm em r.v, um vetor por restrição.
    Sinal: o SciPy escreve L = f + v'c. Com NonlinearConstraint(g, 0, np.inf)
    (g >= 0) e NonlinearConstraint(h, 0, 0):  u_UC = -r.v[0],  lambda_UC = -r.v[1].
    A produção com scipy.optimize.linprog (como no 1.1): os preços-sombra são
    -res.ineqlin.marginals (o SciPy minimiza -L e dá derivadas em ordem a b).

Usa kkt_check.py (nesta pasta). Os slides usam vírgula decimal; aqui o ponto.
Não há função da UC neste deck: é um exemplo. Implementação didática.
```

In [3]:
import warnings

import numpy as np
from scipy.optimize import NonlinearConstraint, linprog, minimize

from kkt_check import kkt_check

# Com restrições lineares, o quasi-Newton do trust-constr avisa que a Hessiana
# da restrição é nula («delta_grad == 0.0»); é esperado e não afeta o resultado.
warnings.filterwarnings("ignore", message="delta_grad == 0.0")


def confere(v, s, d):
    """v confere com o valor s do slide, mostrado com d casas decimais?"""
    v = np.ravel(np.asarray(v, float)); s = np.ravel(np.asarray(s, float))
    d = np.ravel(np.asarray(d, float))
    return bool(np.all(np.abs(v - s) <= 0.5 * 10.0 ** (-d) * (1 + 1e-6)))


def perto(v, s, tol):
    """v está a menos de tol do valor exato s?"""
    return bool(np.all(np.abs(np.ravel(np.asarray(v, float))
                              - np.ravel(np.asarray(s, float))) <= tol))


def simnao(c):
    return tuple("sim" if ci else "não" for ci in c)


sn = lambda b: "sim" if b else "não"
vec = lambda v: ", ".join("%.4f" % (t + 0.0) for t in np.ravel(v))
ok = True
tolA = 1e-10        # contas exatas (a)
tolB = 1e-4         # solver (b)

print("Otimização — deck 4.2: condições de Karush-Kuhn-Tucker")
print("Convenção da UC: g_j >= 0, h_l = 0, L = f - u'g - lambda'h  (nas aulas: u = lambda_j, lambda = beta_l)")
print("\n=============== (a) pelas condições KKT ===============")

# ------------------------------------------------ 1. De 4.1 para 4.2
df = lambda x: 2 * (np.asarray(x) - 1)
dg = lambda x: np.array([[-2 * x[0], -2 * x[1]]])        # grad g' para g = R2 - x1^2 - x2^2
ucalc = lambda x: np.linalg.lstsq(dg(x).T, df(x), rcond=None)[0]   # grad f = u grad g
print("\n1. min (x1-1)^2 + (x2-1)^2  s.a.  g = R^2 - x1^2 - x2^2 >= 0")
x = np.array([1.0, 1.0]); g = 4 - x @ x
c1, r = kkt_check(df(x), [g], dg(x), [0.0], tol=tolA)
print("   largo (R^2 = 4):    x* = (1, 1), g = %g > 0 (inativa), u* = 0, grad f = 0;  KKT: %s"
      % (g, sn(c1)))
x = np.array([1.0, 1.0]) / np.sqrt(2); g = 1 - x @ x; uap = ucalc(x)
c2, r = kkt_check(df(x), [g], dg(x), uap, tol=tolA)
print("   apertado (R^2 = 1): x* = (%.4f, %.4f), g = %.1e (ativa), u* = %.4f = sqrt(2) - 1;  KKT: %s"
      % (*x, g, uap[0], sn(c2)))
x = np.array([1.0, 1.0]) * np.sqrt(2); g = 4 - x @ x; u = ucalc(x)
c3, r = kkt_check(df(x), [g], dg(x), u, tol=tolA)
print("   Geometria (2): x = (sqrt 2, sqrt 2) no largo: u = %.4f = 1/sqrt(2) - 1 < 0;  KKT: %s "
      "(falha o sinal: %.4f)" % (u[0], sn(c3), r.sinal))
c = [c1, c2, perto(uap, np.sqrt(2) - 1, tolA), perto(u, 1 / np.sqrt(2) - 1, tolA), not c3]
print("   largo KKT: %s | apertado KKT: %s | u* = sqrt2 - 1: %s | u = 1/sqrt2 - 1: %s"
      " | (sqrt2, sqrt2) não é KKT: %s" % simnao(c))
ok = ok and all(c)

# ------------------------------------------------ 2. Exemplo 1: 2^J casos
f1 = lambda x: (x[0] - 2)**2 + 2 * (x[1] - 1)**2
df1 = lambda x: np.array([2 * (x[0] - 2), 4 * (x[1] - 1)])
g1 = lambda x: np.array([3 - x[0] - 4 * x[1], x[0] - x[1]])
Jg1 = np.array([[-1.0, -4], [1, -1]])
print("\n2. Exemplo 1: min (x1-2)^2 + 2(x2-1)^2  s.a.  g1 = 3 - x1 - 4x2 >= 0,  g2 = x1 - x2 >= 0")
print("   2(x1-2) + u1 - u2 = 0,  4(x2-1) + 4u1 + u2 = 0,  u1 g1 = 0,  u2 g2 = 0,  u >= 0")
casos = [(0, 0), (0, 1), (1, 0), (1, 1)]      # 1 = restrição ativa (g_j = 0); 0 = u_j = 0
hip = ["u1 = u2 = 0     ", "u1 = 0, g2 = 0  ", "g1 = 0, u2 = 0  ", "g1 = g2 = 0     "]
Xs = [(2, 1), (4/3, 4/3), (5/3, 1/3), (3/5, 3/5)]          # slide
Us = [(0, 0), (0, -4/3), (2/3, 0), (22/25, -48/25)]


def caso_ex1(a1, a2, b=3.0):
    """Sistema linear do caso (a1, a2) em (x1, x2, u1, u2)."""
    M = [[2, 0, 1, -1], [0, 4, 4, 1]]; rhs = [4, 4]        # estacionariedade
    M.append([1, 4, 0, 0] if a1 else [0, 0, 1, 0]); rhs.append(b if a1 else 0)
    M.append([1, -1, 0, 0] if a2 else [0, 0, 0, 1]); rhs.append(0)
    z = np.linalg.solve(np.array(M, float), np.array(rhs, float))
    return z[:2], z[2:]


print("   %-5s %-17s %-19s %-19s %-16s %s" % ("caso", "hipótese", "x", "u", "g", "verificação"))
c = []
for k, (a1, a2) in enumerate(casos):
    x, u = caso_ex1(a1, a2); g = g1(x)
    ver = ["g%d < 0 (inadmissível)" % (j + 1) for j in range(2) if g[j] < -tolA]
    ver += ["u%d < 0" % (j + 1) for j in range(2) if u[j] < -tolA]
    ver = "; ".join(ver) + "  x" if ver else "ponto KKT  V"
    print("   %-5d %s (%7.4f,%7.4f) (%7.4f,%7.4f) (%6.3f,%6.3f) %s"
          % (k + 1, hip[k], *x, *(u + 0.0), *g, ver))
    c += [perto(x, Xs[k], tolA), perto(u, Us[k], tolA)]
    if k == 2:
        x3, u3 = x, u
cK, r = kkt_check(df1(x3), g1(x3), Jg1, u3, tol=tolA)
gx3 = g1(x3)
print("   Sobrevive o caso 3: x* = (5/3, 1/3), f* = %.4f; ativas I = {%s}; u1 g1 = %.1e, u2 g2 = %.1e"
      % (f1(x3), ", ".join(map(str, r.ativas)), *(u3 * gx3)))
print("   Geometria (2): grad f(x*) = (%.4f, %.4f) = u1 grad g1 = (2/3)(-1,-4)" % tuple(df1(x3)))
c = [all(c), cK, perto(f1(x3), 1, tolA), perto(gx3[1], 4/3, tolA),
     perto(df1(x3), [-2/3, -8/3], tolA)]
print("   tabela dos 4 casos: %s | caso 3 é KKT: %s | f* = 1: %s | g2 = 4/3: %s"
      " | grad f = (2/3)(-1,-4): %s" % simnao(c))
ok = ok and all(c)

# ------------------------------------------------ 3. Ativa com u = 0
print("\n3. min x^2  s.a.  g = x >= 0:  x* = 0, g(x*) = 0 (ativa), 2x - u = 0 => u* = 0")
cK, r = kkt_check([0.0], [0.0], [[1.0]], [0.0], tol=tolA)
print("   KKT: %s; ativas I = {%s} com u = 0 (ativa não implica u > 0)"
      % (sn(cK), ", ".join(map(str, r.ativas))))
ok = ok and cK and r.ativas == [1]

# ------------------------------------------------ 4. Exemplo 2
f2 = lambda x: (x[0] - 3)**2 + (x[1] - 2)**2
df2 = lambda x: np.array([2 * (x[0] - 3), 2 * (x[1] - 2)])
g2 = lambda x: np.array([5 - x[0]**2 - x[1]**2, 4 - x[0] - 2 * x[1], x[0], x[1]])
Jg2 = lambda x: np.array([[-2 * x[0], -2 * x[1]], [-1, -2], [1, 0], [0, 1]], float)
print("\n4. Exemplo 2: min (x1-3)^2 + (x2-2)^2  s.a.  5 - x1^2 - x2^2 >= 0, 4 - x1 - 2x2 >= 0,"
      " x1 >= 0, x2 >= 0")
x = np.array([2.0, 1.0]); g = g2(x); J = Jg2(x)
I = np.flatnonzero(np.abs(g) <= tolA)                       # ativas
u = np.zeros(4)
u[I] = np.linalg.solve(J[I].T, df2(x))                       # grad f = sum_{j em I} u_j grad g_j
cK, r = kkt_check(df2(x), g, J, u, tol=tolA)
rk = np.linalg.matrix_rank(J[I])
print("   x = (2, 1): g = (%s), ativas I = {%s}; grad f = (%g, %g), grad g1 = (%g, %g), grad g2 = (%g, %g)"
      % (" ".join("%g" % t for t in g), ", ".join(str(j + 1) for j in I), *df2(x), *J[0], *J[1]))
print("   u = (%s)  (slide: 1/3, 2/3, 0, 0);  LICQ: característica de [grad g1 grad g2] = %d;"
      "  f = %g;  KKT: %s" % (vec(u), rk, f2(x), sn(cK)))
c = [perto(u, [1/3, 2/3, 0, 0], tolA), rk == 2, perto(f2(x), 2, tolA), cK]
print("   u1 = 1/3, u2 = 2/3: %s | LICQ: %s | f = 2: %s | ponto KKT: %s" % simnao(c))
ok = ok and all(c)

# ------------------------------------------------ 5. KKT dá candidatos
print("\n5. min -x^2  s.a.  g1 = x + 1 >= 0,  g2 = 2 - x >= 0  (os 4 casos; «ambas ativas» é impossível)")
fm = lambda x: -x**2
gm = lambda x: np.array([x + 1, 2 - x])
Jgm = np.array([[1.0], [-1.0]])
c = []
for x, u, nome in [(0.0, (0, 0), "máximo"), (-1.0, (2, 0), "mínimo local"),
                   (2.0, (0, 4), "mínimo global")]:
    cK, r = kkt_check([-2 * x], gm(x), Jgm, u, tol=tolA)
    print("   x = %2g: u = (%g, %g), f = %2g, f'' = -2;  KKT: %s -> %s" % (x, *u, fm(x) + 0.0, sn(cK), nome))
    c.append(cK)
cand = [0.0, -1.0, 2.0]
c.append(cand[int(np.argmin([fm(t) for t in cand]))] == 2)
print("   os três são pontos KKT: %s | o menor f (global) é x = 2: %s" % (sn(all(c[:3])), sn(c[3])))
ok = ok and all(c)

# ------------------------------------------------ 6. O que medem os multiplicadores
print("\n6. Sensibilidade no exemplo 1: g1 = b - x1 - 4x2 >= 0, b = 3 -> 3.1 (caso 3: g1 ativa, u2 = 0)")
fb = lambda b: f1(caso_ex1(1, 0, b)[0])
f31 = fb(3.1)
print("   f*(3) = %.4f, f*(3.1) = %.5f (exato: (6 - b)^2/9); aproximação 1 - u1*0.1 = %.4f"
      % (fb(3.0), f31, 1 - 2 / 3 * 0.1))
c = [perto(fb(3.0), 1, tolA), confere(f31, 0.934, 3), confere(1 - 2 / 3 * 0.1, 0.933, 3),
     perto(f31, (6 - 3.1)**2 / 9, tolA)]
print("   f*(3) = 1: %s | f*(3.1) = 0.934: %s | aproximação 0.933: %s | (6-b)^2/9: %s" % simnao(c))
ok = ok and all(c)

# produção do 1.1: max 40x1 + 30x2  <=>  min -(40x1 + 30x2)
print("\n   Produção do 1.1: max 40x1 + 30x2 s.a. 2x1 + x2 <= 100 (máquina), x1 + x2 <= 80 (mão de obra),")
print("   x1 <= 40 (procura), x >= 0.  Em g_j >= 0: g = b - A x >= 0 e x >= 0;  f = -(40x1 + 30x2)")
Ap = np.array([[2.0, 1], [1, 1], [1, 0]]); bp = np.array([100.0, 80, 40]); cp = np.array([40.0, 30])
x = np.array([20.0, 60.0])
gp = np.concatenate([bp - Ap @ x, x]); Jgp = np.vstack([-Ap, np.eye(2)])
I = np.flatnonzero(np.abs(gp) <= tolA)
u = np.zeros(5)
u[I] = np.linalg.solve(Jgp[I].T, -cp)                        # grad f = -c = sum u_j grad g_j
cK, r = kkt_check(-cp, gp, Jgp, u, tol=tolA)
print("   x* = (20, 60), L* = %g;  ativas I = {%s} (máquina, mão de obra): 2u1 + u2 = 40, u1 + u2 = 30"
      % (cp @ x, ", ".join(str(j + 1) for j in I)))
print("   u = (%s)  -> preços-sombra 10 EUR/h, 20 EUR/h, 0;  KKT: %s" % (vec(u), sn(cK)))
c = [perto(cp @ x, 2600, tolA), perto(u, [10, 20, 0, 0, 0], tolA), cK]
print("   L* = 2600: %s | u = (10, 20, 0): %s | ponto KKT: %s" % simnao(c))
ok = ok and all(c)

# ------------------------------------------------ 7. Quando KKT falha: LICQ
print("\n7. min -x1  s.a.  g1 = (1 - x1)^3 - x2 >= 0,  g2 = x2 >= 0;  x* = (1, 0)")
x = np.array([1.0, 0.0]); gf = np.array([-1.0, 0.0])
J = np.array([[-3 * (1 - x[0])**2 + 0.0, -1], [0, 1]])
u = np.linalg.lstsq(J.T, gf, rcond=None)[0]                  # melhor u (mínimos quadrados)
resmin = np.linalg.norm(gf - J.T @ u)
print("   grad g1 = (%g, %g), grad g2 = (%g, %g): característica %d (dependentes)"
      % (*J[0], *J[1], np.linalg.matrix_rank(J)))
print("   min_u |grad f - J'u| = %.4f: a componente x1 de grad f = (-1, 0) não se obtém -> não há u" % resmin)
c = [np.linalg.matrix_rank(J) == 1, perto(resmin, 1, tolA)]
print("   gradientes dependentes: %s | sem multiplicadores (resíduo 1): %s" % simnao(c))
ok = ok and all(c)

# ------------------------------------------------ 8. Exercícios
print("\n8. Exercícios")
# Ex. 1: min -ln(x1+1) - x2 s.a. 3 - 2x1 - x2 >= 0, x1 >= 0, x2 >= 0
fe1 = lambda x: -np.log(x[0] + 1) - x[1]
dfe1 = lambda x: np.array([-1 / (x[0] + 1), -1.0])
ge1 = lambda x: np.array([3 - 2 * x[0] - x[1], x[0], x[1]])
Jge1 = np.array([[-2.0, -1], [1, 0], [0, 1]])
x = np.array([0.0, 3.0]); g = ge1(x); I = np.flatnonzero(np.abs(g) <= tolA)
u = np.zeros(3); u[I] = np.linalg.solve(Jge1[I].T, dfe1(x))
cK1, r = kkt_check(dfe1(x), g, Jge1, u, tol=tolA)
print("   Ex. 1: x* = (0, 3), ativas I = {%s}, u = (%s), f* = %.4f;  KKT: %s"
      % (", ".join(str(j + 1) for j in I), vec(u), fe1(x), sn(cK1)))
c = [list(I + 1) == [1, 2], cK1]
# Ex. 2: min x1^2 + x2^2 s.a. 5 - x1^2 - x2^2 >= 0, x1 + 2x2 = 4, x >= 0
fe2 = lambda x: x[0]**2 + x[1]**2
dfe2 = lambda x: 2 * np.asarray(x, float)
ge2 = lambda x: np.array([5 - x[0]**2 - x[1]**2, x[0], x[1]])
Jge2 = lambda x: np.array([[-2 * x[0], -2 * x[1]], [1, 0], [0, 1]], float)
he2 = lambda x: np.array([x[0] + 2 * x[1] - 4])
Jhe2 = np.array([[1.0, 2.0]])
x = np.array([0.8, 1.6]); g = ge2(x)
lam = np.linalg.lstsq(Jhe2.T, dfe2(x), rcond=None)[0]        # só a igualdade é ativa: u = 0
cK2, r = kkt_check(dfe2(x), g, Jge2(x), np.zeros(3), he2(x), Jhe2, lam, tol=tolA)
print("   Ex. 2: x* = (4/5, 8/5), g = (%s) > 0 (inativas, u = 0), lambda = %.4f (slide: 8/5);  KKT: %s"
      % (vec(g), lam[0], sn(cK2)))
c += [r.ativas == [], perto(lam, 8 / 5, tolA), cK2]
# Ex. 4: a lata com h <= 6 (g = 6 - h >= 0), z = (r, h)
V = 330.0
A = lambda z: 2 * np.pi * z[0]**2 + 2 * np.pi * z[0] * z[1]
dA = lambda z: np.array([4 * np.pi * z[0] + 2 * np.pi * z[1], 2 * np.pi * z[0]])
hv = lambda z: np.array([np.pi * z[0]**2 * z[1] - V])
Jhv = lambda z: np.array([[2 * np.pi * z[0] * z[1], np.pi * z[0]**2]])
gh = lambda z: np.array([6 - z[1]])
Jgh = np.array([[0.0, -1.0]])
hh = 6.0; rr = np.sqrt(V / (6 * np.pi)); z = np.array([rr, hh])
lam = (2 * rr + hh) / (rr * hh)                              # de dL/dr = 0
u = lam * np.pi * rr**2 - 2 * np.pi * rr                     # de dL/dh = 0: 2 pi r - lambda pi r^2 + u = 0
cK4, r = kkt_check(dA(z), gh(z), Jgh, [u], hv(z), Jhv(z), [lam], tol=1e-9)
print("   Ex. 4: h* = 6, r* = %.4f cm, A* = %.4f cm^2, lambda = %.4f, u* = %.4f cm^2/cm;  KKT: %s"
      % (rr, A(z), lam, u, sn(cK4)))
c += [confere(rr, 4.18, 2), confere(A(z), 267.7, 1), confere(u, 5.19, 2), cK4]
print("   Ex. 1 ativas {1,2}: %s | KKT: %s | Ex. 2 inativas: %s | lambda = 8/5: %s | KKT: %s"
      " | Ex. 4 r* = 4.18: %s | A* = 267.7: %s | u* = 5.19: %s | KKT: %s" % simnao(c))
ok = ok and all(c)

# =================================================== (b) com um solver
print("\n=============== (b) com um solver ===============")
print("Solver: scipy.optimize.minimize(method='trust-constr'), multiplicadores em r.v.")
print("O SciPy escreve L = f + v'c: com NonlinearConstraint(g, 0, np.inf) e NonlinearConstraint(h, 0, 0),")
print("u_UC = -r.v[0] e lambda_UC = -r.v[1]  (fmincon: u_UC = lambda.ineqnonlin, lambda_UC = -lambda.eqnonlin).")
opts = dict(gtol=1e-10, xtol=1e-12, maxiter=3000)
# f, grad f, g (>= 0), jac g, h (= 0) ou None, jac h, x0, x* exato, u* exato, lambda* exato, nome
probs = [(f1, df1, g1, lambda x: Jg1, None, None, [0, 0], [5/3, 1/3], [2/3, 0], None, "Exemplo 1"),
         (f2, df2, g2, Jg2, None, None, [0.5, 0.5], [2, 1], [1/3, 2/3, 0, 0], None, "Exemplo 2"),
         (fe2, dfe2, ge2, Jge2, he2, lambda x: Jhe2, [1, 1], [0.8, 1.6], [0, 0, 0], [1.6],
          "Exercício 2"),
         (A, dA, gh, lambda z: Jgh, hv, Jhv, [3, 6], None, None, None, "Exercício 4 (lata, h <= 6)")]
for fo, dfo, gc, Jg, hc, Jh, x0, xs, us, ls, nome in probs:
    cons = [NonlinearConstraint(gc, 0, np.inf, jac=Jg)]
    if hc is not None:
        cons.append(NonlinearConstraint(hc, 0, 0, jac=Jh))
    r = minimize(fo, x0, jac=dfo, method="trust-constr", constraints=cons, options=opts)
    u = -r.v[0]
    lam = -r.v[1] if hc is not None else None
    print("   %s: x = (%s), f = %.4f, u = -r.v[0] = (%s)%s  [status %d, %d it., %d aval. de f]"
          % (nome, vec(r.x), r.fun, vec(u),
             "" if lam is None else ", lambda = -r.v[1] = (%s)" % vec(lam), r.status, r.nit, r.nfev))
    if xs is None:      # exercício 4: comparar com os valores do slide
        c = [confere(r.x[1], 6, 2), confere(r.x[0], 4.18, 2), confere(r.fun, 267.7, 1),
             confere(u, 5.19, 2)]
        print("      h* = 6: %s | r* = 4.18: %s | A* = 267.7: %s | u* = 5.19: %s" % simnao(c))
    else:
        c = [perto(r.x, xs, tolB), perto(u, us, tolB)]
        if ls is not None:
            c.append(perto(lam, ls, tolB))
        print("      x*: %s | u*: %s" % simnao(c[:2]) + ("" if ls is None else " | lambda*: %s" % sn(c[2])))
    ok = ok and all(c)

# produção (PL) com linprog, como no 1.1
res = linprog(c=-cp, A_ub=Ap, b_ub=bp, bounds=[(0, None)] * 2)
u = -res.ineqlin.marginals
print("   Produção com linprog (min -L): x = (%s), L = %.4f, res.ineqlin.marginals = (%s)"
      % (vec(res.x), -res.fun, vec(res.ineqlin.marginals)))
print("      preços-sombra = -marginals = (%s)" % vec(u))
c = [perto(res.x, [20, 60], tolB), perto(-res.fun, 2600, tolB), perto(u, [10, 20, 0], tolB)]
print("      x* = (20, 60): %s | L* = 2600: %s | preços-sombra (10, 20, 0): %s" % simnao(c))
ok = ok and all(c)

print("\nconfere com os slides: %s" % ("sim" if ok else "não"))


Otimização — deck 4.2: condições de Karush-Kuhn-Tucker
Convenção da UC: g_j >= 0, h_l = 0, L = f - u'g - lambda'h  (nas aulas: u = lambda_j, lambda = beta_l)

=============== (a) pelas condições KKT ===============

1. min (x1-1)^2 + (x2-1)^2  s.a.  g = R^2 - x1^2 - x2^2 >= 0
   largo (R^2 = 4):    x* = (1, 1), g = 2 > 0 (inativa), u* = 0, grad f = 0;  KKT: sim
   apertado (R^2 = 1): x* = (0.7071, 0.7071), g = 2.2e-16 (ativa), u* = 0.4142 = sqrt(2) - 1;  KKT: sim
   Geometria (2): x = (sqrt 2, sqrt 2) no largo: u = -0.2929 = 1/sqrt(2) - 1 < 0;  KKT: não (falha o sinal: 0.2929)
   largo KKT: sim | apertado KKT: sim | u* = sqrt2 - 1: sim | u = 1/sqrt2 - 1: sim | (sqrt2, sqrt2) não é KKT: sim

2. Exemplo 1: min (x1-2)^2 + 2(x2-1)^2  s.a.  g1 = 3 - x1 - 4x2 >= 0,  g2 = x1 - x2 >= 0
   2(x1-2) + u1 - u2 = 0,  4(x2-1) + 4u1 + u2 = 0,  u1 g1 = 0,  u2 g2 = 0,  u >= 0
   caso  hipótese          x                   u                   g                verificação
   1     u1 = u2 = 0      ( 2.00

## 4.3.1 Método da função de penalização exterior

Exemplo [`ex04_3_1_exterior_penalty.py`](https://github.com/aguilarmadeira/optimization-course/blob/main/code/python/04_constrained_optimization/04_3_1_exterior_penalty/ex04_3_1_exterior_penalty.py) · funções da UC: [`nelder_mead_comp.py`](https://github.com/aguilarmadeira/optimization-course/blob/main/code/python/common/nelder_mead_comp.py), [`penalty_exterior.py`](https://github.com/aguilarmadeira/optimization-course/blob/main/code/python/04_constrained_optimization/04_3_1_exterior_penalty/penalty_exterior.py), [`nelder_mead.py`](https://github.com/aguilarmadeira/optimization-course/blob/main/code/python/03_multivariate_optimization/03_2_3_nelder_mead/nelder_mead.py)

```text
Reproduz os exemplos do deck 4.3.1 (penalização exterior).

1. «Ideia»: min x^2 s.a. x - 1 >= 0; x_R = R/(1+R): 1/2, 0.909 (R = 1, 10).
2. «O exemplo nas contas» e «As KKT reaparecem»: min x1^2 + x2^2 s.a.
   x1 + x2 - 1 >= 0, R = 1, 10, 100, 1000: x_R, g(x_R), ||x_R - x*|| = O(1/R), u_R.
3. Restrição inativa: min (x1-1)^2 + (x2-1)^2 s.a. 9 - x1^2 - x2^2 >= 0:
   x_R = (1,1) e u_R = 0 para todo o R.
4. «A escala das restrições importa»: g1 = 1 - x1, g2 = 100(1 - x2), R = 1, 10, 100.
5. «E com igualdades?»: min 2x1^2 + x2^2 s.a. x1 + x2 - 1 = 0: x_R, h(x_R), beta_R
   (o multiplicador beta das aulas, L = f - beta h; em 4.1, lambda* = -4/3).
6. A chamada do slide «No computador» (tolviol = tolx = 1e-6, tmax = 12).
7. A comparação do deck 4.3.2 (lado exterior): de (0,0), 6 ciclos, n_f = 844;
   de (1,1), 6 ciclos, n_f = 864 (até ||x^(t) - x*|| <= 1e-4, sem a avaliação final).

Minimizador interno: o Nelder–Mead da UC na versão do script da comparação
(nelder_mead_comp, tolx = 1e-10, tolf = 1e-12, kmax = 2000; ver o cabeçalho
de nelder_mead_comp.py e o README). As tabelas 2-5 são uma única corrida
com arranque a quente (R^(0) = 1, c = 10), lida em history.
Os slides usam vírgula decimal; aqui usa-se o ponto.
```

In [4]:
import numpy as np

from nelder_mead_comp import nelder_mead_comp
from penalty_exterior import penalty_exterior



def confere(v, s, d):
    """v confere com o valor s do slide, mostrado com d casas decimais?"""
    v = np.ravel(np.asarray(v, float)); s = np.ravel(np.asarray(s, float))
    d = np.ravel(np.asarray(d, float))
    return bool(np.all(np.abs(v - s) <= 0.5 * 10.0 ** (-d) * (1 + 1e-6)))


def simnao(c):
    return tuple("sim" if ci else "não" for ci in c)


NM = lambda P, x: nelder_mead_comp(P, x, 1e-10, 1e-12, 2000)   # interno dos exemplos
sem = lambda x: []                                              # sem restrições deste tipo
ok = True
print("Otimização — deck 4.3.1: penalização exterior")

# ------------------------------------------------------------ 1. Ideia (1D)
r = penalty_exterior(lambda x: x[0]**2, lambda x: x[0] - 1, sem, [0.0], 1, 10, 0, 0, 2, NM)
xR = r.history[1:, 5]
print("\n1. min x^2 s.a. x - 1 >= 0: x_R = R/(1+R) (R = 0: x_R = 0, o mínimo livre)")
for Ri, xi in zip(r.history[1:, 1], xR):
    print("   R = %4g: x_R = %.4f  (exato %.4f)" % (Ri, xi, Ri / (1 + Ri)))
c = [confere(xR, [0.5, 0.909], 3)]
print("  x_R = 1/2, 0.909: %s" % simnao(c))
ok = ok and all(c)

# ------------------------------------------------------------ 2. O exemplo nas contas
f = lambda x: x[0]**2 + x[1]**2
g = lambda x: x[0] + x[1] - 1
xs = np.array([0.5, 0.5])
r = penalty_exterior(f, g, sem, [0, 0], 1, 10, 0, 0, 4, NM)
H = r.history[1:]
Rv = H[:, 1]; X = H[:, 5:7]; u = H[:, 7]
gv = X.sum(axis=1) - 1; err = np.linalg.norm(X - xs, axis=1)
print("\n2. min x1^2 + x2^2 s.a. x1 + x2 - 1 >= 0, de (0,0), R^(0) = 1, c = 10 (x* = (1/2,1/2), u* = 1)")
print("  %6s %20s %10s %12s %12s %8s %8s" % ("R", "x_R", "g(x_R)", "||x_R-x*||", "R*||x_R-x*||", "u_R", "n_f"))
for i in range(4):
    print("  %6g   (%.4f; %.4f) %10.4f %12.5f %12.4f %8.4f %8d"
          % (Rv[i], X[i, 0], X[i, 1], gv[i], err[i], Rv[i] * err[i], u[i], H[i, 2]))
print("  erro O(1/R): R*||x_R - x*|| -> 1/(2 sqrt 2) = %.4f;  u_R = 2R/(1+2R) -> 1" % (1 / (2 * np.sqrt(2))))
c = [confere(X, np.repeat([0.333, 0.476, 0.4975, 0.4998], 2).reshape(4, 2),
             np.repeat([3, 3, 4, 4], 2).reshape(4, 2)),
     confere(gv, [-0.333, -0.0476, -0.0050, -0.0005], [3, 4, 4, 4]),
     confere(err, [0.236, 0.0337, 0.0035, 0.00035], [3, 4, 4, 5]),
     confere(u, [0.667, 0.952, 0.995, 0.9995], [3, 3, 3, 4])]
print("  x_R: %s | g(x_R): %s | ||x_R - x*||: %s | u_R: %s" % simnao(c))
ok = ok and all(c)

# ------------------------------------------------------------ 3. Restrição inativa
r = penalty_exterior(lambda x: (x[0] - 1)**2 + (x[1] - 1)**2, lambda x: 9 - x[0]**2 - x[1]**2,
                     sem, [0, 0], 1, 10, 0, 0, 4, NM)
H = r.history[1:]
print("\n3. Restrição inativa: min (x1-1)^2 + (x2-1)^2 s.a. 9 - x1^2 - x2^2 >= 0")
for row in H:
    print("   R = %4g: x_R = (%.4f; %.4f), g = %.4f, u_R = %.4f"
          % (row[1], row[5], row[6], 9 - row[5]**2 - row[6]**2, row[7] + 0.0))
print("   (%s: x não mudou e a violação é 0)" % r.message)
c = [confere(H[:, 5:7], np.ones_like(H[:, 5:7]), 4), confere(H[:, 7], 0 * H[:, 7], 4)]
print("  x_R = (1,1): %s | u_R = 0: %s" % simnao(c))
ok = ok and all(c)

# ------------------------------------------------------------ 4. Escala das restrições
fe = lambda x: (x[0] - 2)**2 + (x[1] - 2)**2
r = penalty_exterior(fe, lambda x: [1 - x[0], 100 * (1 - x[1])], sem, [0, 0], 1, 10, 0, 0, 3, NM)
H = r.history[1:]
kap = (2 + 2e4 * H[:, 1]) / (2 + 2 * H[:, 1])     # Hessiana de P em x_R: diag(2+2R, 2+2e4 R)
print("\n4. Escala: min (x1-2)^2 + (x2-2)^2 s.a. 1 - x1 >= 0 e 100(1 - x2) >= 0 (x* = (1,1))")
print("  %6s %10s %12s %10s" % ("R", "x1_R", "x2_R", "kappa"))
for i in range(3):
    print("  %6g %10.4f %12.7f %10.0f" % (H[i, 1], H[i, 5], H[i, 6], kap[i]))
r2 = penalty_exterior(fe, lambda x: [1 - x[0], 1 - x[1]], sem, [0, 0], 1, 10, 0, 0, 1, NM)
print("  com g2 = 1 - x2 e R = 1: x_R = (%.4f; %.4f) = ((2+R)/(1+R), (2+R)/(1+R)), kappa = 1"
      % tuple(r2.x))
c = [confere(H[:, 5], [1.50, 1.091, 1.0099], [2, 3, 4]),
     confere(H[:, 6], [1.0001, 1.00001, 1.000001], [4, 5, 6]),
     confere(kap, [5000, 9091, 9901], 0), confere(r2.x, [1.5, 1.5], 4)]
print("  x1_R: %s | x2_R: %s | kappa: %s | com g2: %s" % simnao(c))
ok = ok and all(c)

# ------------------------------------------------------------ 5. Igualdades
r = penalty_exterior(lambda x: 2 * x[0]**2 + x[1]**2, sem, lambda x: x[0] + x[1] - 1,
                     [0, 0], 1, 10, 0, 0, 4, NM)
H = r.history[1:]
X = H[:, 5:7]; hv = X.sum(axis=1) - 1; lam = H[:, 7]
print("\n5. Igualdade: min 2x1^2 + x2^2 s.a. x1 + x2 - 1 = 0 (x* = (1/3, 2/3), lambda* = 4/3)")
for i in range(4):
    print("   R = %4g: x_R = (%.4f; %.4f), h = %.4f, beta_R = -2Rh = %.3f"
          % (H[i, 1], X[i, 0], X[i, 1], hv[i], lam[i]))
c = [confere(X, [[0.2000, 0.4000], [0.3125, 0.6250], [0.3311, 0.6623], [0.3331, 0.6662]], 4),
     confere(hv, [-0.4000, -0.0625, -0.0066, -0.0007], 4),
     confere(lam, [0.800, 1.250, 1.325, 1.332], 3)]
print("  x_R: %s | h(x_R): %s | beta_R: %s" % simnao(c))
ok = ok and all(c)

# ------------------------------------------------------------ 6. Chamada do slide
print("\n6. Chamada do slide: penalty_exterior(f, g, h, [0, 0], 0.1, 10, 1e-6, 1e-6, 12)")
print("   (interno por omissão: nelder_mead_comp; o slide usa o fminsearch do MATLAB)")
r = penalty_exterior(f, g, sem, [0, 0], 0.1, 10, 1e-6, 1e-6, 12, verbose=True)
nfc = r.history[1:, 2]
print("   nit = %d (R = %g), u = %.4f, n_f por ciclo %d-%d, nfev = %d (inclui f(x) final)"
      % (r.nit, r.R, r.u[0], nfc.min(), nfc.max(), r.nfev))
print("   %s" % r.message)
H = r.history
print("   nota: com x_R exatos o critério valeria no ciclo 9 (viol = 5e-8, passo = %.1e); aqui o passo"
      % (np.sqrt(2) * (1e7 / (1 + 2e7) - 1e6 / (1 + 2e6))))
print("         no ciclo 9 é %.1e > 1e-6 (erro do interno ao longo de g = 0, P mal condicionada)"
      % (np.linalg.norm(H[9, 5:7] - H[8, 5:7]) / max(1, np.linalg.norm(H[8, 5:7]))))
print("   slide (fminsearch do MATLAB): nit = 10 (R = 1e8); u ~ 1; nfev ~ 130-210 por ciclo")
c = [abs(r.u[0] - 1) <= 0.01]
print("  u ~ 1 (a menos de 1 %%): %s | nit = 10: %s (informativo: depende do minimizador interno)"
      % (simnao(c)[0], "sim" if r.nit == 10 else "não"))
ok = ok and all(c)

# ------------------------------------------------------------ 7. Comparação (deck 4.3.2)
print("\n7. Comparação exterior vs. interior (deck 4.3.2), lado exterior: R^(0) = 0.1, c = 10,")
print("   tolerâncias internas 1e-10; para no 1.º ciclo com ||x^(t) - x*|| <= 1e-4 (lido em history)")


def ciclos_ate(r, tol=1e-4):
    """1.º ciclo t com ||x^(t) - x*|| <= tol e n_f acumulado até aí (sem a avaliação final)."""
    H = r.history
    e = np.linalg.norm(H[1:, 5:7] - xs, axis=1)
    t = int(np.argmax(e <= tol)) + 1
    return t, int(H[t, 3]), H[1:t + 1, 2]


res = {}
for x0 in ([0.0, 0.0], [1.0, 1.0]):
    r = penalty_exterior(f, g, sem, x0, 0.1, 10, 0, 0, 12, NM)
    t, nf, nfc = ciclos_ate(r)
    res[tuple(x0)] = (t, nf, int(np.median(nfc)))
    print("   de (%g,%g):" % tuple(x0))
    for row in r.history[1:t + 1]:
        print("     t = %d  R = %8.1e  x = (%.5f, %.5f)  ||x - x*|| = %.1e  n_f = %3d  acum. = %d"
              % (row[0], row[1], row[5], row[6], np.linalg.norm(row[5:7] - xs), row[2], row[3]))
    print("     -> %d ciclos, n_f = %d, mediana por ciclo = %d" % res[tuple(x0)])
c = [res[(0.0, 0.0)][:2] == (6, 844), res[(0.0, 0.0)][2] == 140, res[(1.0, 1.0)][:2] == (6, 864)]
print("  (0,0): 6 ciclos, n_f = 844: %s | mediana 140 (~140 por ciclo): %s | (1,1): 6 ciclos, n_f = 864: %s"
      % simnao(c))
ok = ok and all(c)

# informativo: o mesmo com o nelder_mead do deck 3.2.3 (contração exterior aceite com <=)
from nelder_mead import nelder_mead  # noqa: E402  (deck 3.2.3)
NM3 = lambda P, x: nelder_mead(P, x, 1e-10, 1e-12, 2000)
v = [ciclos_ate(penalty_exterior(f, g, sem, x0, 0.1, 10, 0, 0, 12, NM3))[:2]
     for x0 in ([0, 0], [1, 1])]
print("   informativo — com nelder_mead do deck 3.2.3 (aceita a contração exterior com <=):")
print("   (0,0): %d ciclos, n_f = %d;  (1,1): %d ciclos, n_f = %d" % (*v[0], *v[1]))

print("\nconfere com os slides: %s" % ("sim" if ok else "não"))


Otimização — deck 4.3.1: penalização exterior

1. min x^2 s.a. x - 1 >= 0: x_R = R/(1+R) (R = 0: x_R = 0, o mínimo livre)
   R =    1: x_R = 0.5000  (exato 0.5000)
   R =   10: x_R = 0.9091  (exato 0.9091)
  x_R = 1/2, 0.909: sim

2. min x1^2 + x2^2 s.a. x1 + x2 - 1 >= 0, de (0,0), R^(0) = 1, c = 10 (x* = (1/2,1/2), u* = 1)
       R                  x_R     g(x_R)   ||x_R-x*|| R*||x_R-x*||      u_R      n_f
       1   (0.3333; 0.3333)    -0.3333      0.23570       0.2357   0.6667      155
      10   (0.4762; 0.4762)    -0.0476      0.03367       0.3367   0.9524      137
     100   (0.4975; 0.4975)    -0.0050      0.00352       0.3518   0.9950      138
    1000   (0.4998; 0.4998)    -0.0005      0.00035       0.3534   0.9995      144
  erro O(1/R): R*||x_R - x*|| -> 1/(2 sqrt 2) = 0.3536;  u_R = 2R/(1+2R) -> 1
  x_R: sim | g(x_R): sim | ||x_R - x*||: sim | u_R: sim

3. Restrição inativa: min (x1-1)^2 + (x2-1)^2 s.a. 9 - x1^2 - x2^2 >= 0
   R =    1: x_R = (1.0000; 1.0000), g = 7.0000, u

   de (1,1):
     t = 1  R =  1.0e-01  x = (0.08333, 0.08333)  ||x - x*|| = 5.9e-01  n_f = 154  acum. = 154
     t = 2  R =  1.0e+00  x = (0.33333, 0.33333)  ||x - x*|| = 2.4e-01  n_f = 144  acum. = 298
     t = 3  R =  1.0e+01  x = (0.47619, 0.47619)  ||x - x*|| = 3.4e-02  n_f = 136  acum. = 434
     t = 4  R =  1.0e+02  x = (0.49751, 0.49751)  ||x - x*|| = 3.5e-03  n_f = 137  acum. = 571
     t = 5  R =  1.0e+03  x = (0.49975, 0.49975)  ||x - x*|| = 3.5e-04  n_f = 146  acum. = 717
     t = 6  R =  1.0e+04  x = (0.49998, 0.49997)  ||x - x*|| = 3.5e-05  n_f = 147  acum. = 864
     -> 6 ciclos, n_f = 864, mediana por ciclo = 145
  (0,0): 6 ciclos, n_f = 844: sim | mediana 140 (~140 por ciclo): sim | (1,1): 6 ciclos, n_f = 864: sim
   informativo — com nelder_mead do deck 3.2.3 (aceita a contração exterior com <=):
   (0,0): 6 ciclos, n_f = 866;  (1,1): 6 ciclos, n_f = 873

confere com os slides: sim


## 4.3.2 Método da função de barreira / penalização interior

Exemplo [`ex04_3_2_barrier.py`](https://github.com/aguilarmadeira/optimization-course/blob/main/code/python/04_constrained_optimization/04_3_2_barrier/ex04_3_2_barrier.py) · funções da UC: [`barrier_log.py`](https://github.com/aguilarmadeira/optimization-course/blob/main/code/python/04_constrained_optimization/04_3_2_barrier/barrier_log.py), [`nelder_mead_comp.py`](https://github.com/aguilarmadeira/optimization-course/blob/main/code/python/common/nelder_mead_comp.py), [`penalty_exterior.py`](https://github.com/aguilarmadeira/optimization-course/blob/main/code/python/04_constrained_optimization/04_3_1_exterior_penalty/penalty_exterior.py), [`nelder_mead.py`](https://github.com/aguilarmadeira/optimization-course/blob/main/code/python/03_multivariate_optimization/03_2_3_nelder_mead/nelder_mead.py)

```text
Reproduz os exemplos do deck 4.3.2 (barreira logarítmica).

1. «Ideia»: min x^2 s.a. x - 1 >= 0; x_R = (1 + sqrt(1+2R))/2: 1.366, 1.048, 1.005
   (R = 1, 0.1, 0.01).
2. «O exemplo nas contas» e «As KKT reaparecem --- perturbadas»: min x1^2 + x2^2
   s.a. x1 + x2 - 1 >= 0, R = 1, 0.1, 0.01, 0.001: x_R, g(x_R), ||x_R - x*|| = O(R), u_R.
3. Barreira inversa R/g, R = 0.01: x_R = (0.548; 0.548), contra (0.5050; 0.5050) da log.
4. Restrição inativa: min (x1-1)^2 + (x2-1)^2 s.a. 9 - x1^2 - x2^2 >= 0:
   R = 1 -> (0.882; 0.882), R = 0.1 -> (0.986; 0.986); u_R = R/g -> 0.
5. A chamada do slide «No computador» (tolcomp = 1e-8, tolx = 1e-6, tmax = 12).
6. A comparação exterior vs. interior (slide «Exterior vs. interior»): barreira de
   (1,1) em 5 ciclos, 743 chamadas a P, 26 fora de X, n_f = 717; exterior de (0,0)
   em 6 ciclos, n_f = 844, e de (1,1), n_f = 864 (até ||x^(t) - x*|| <= 1e-4, sem a
   avaliação final). Usa penalty_exterior da pasta 04_3_1_exterior_penalty.

Minimizador interno: o Nelder–Mead da UC na versão do script da comparação
(nelder_mead_comp, tolx = 1e-10, tolf = 1e-12, kmax = 2000; ver o cabeçalho
de nelder_mead_comp.py e o README). As tabelas são uma única corrida com
arranque a quente (c = 0.1), lida em history.
Os slides usam vírgula decimal; aqui usa-se o ponto.
```

In [5]:
import numpy as np

from barrier_log import barrier_log
from nelder_mead_comp import nelder_mead_comp
from penalty_exterior import penalty_exterior  # (4.3.1)


def confere(v, s, d):
    """v confere com o valor s do slide, mostrado com d casas decimais?"""
    v = np.ravel(np.asarray(v, float)); s = np.ravel(np.asarray(s, float))
    d = np.ravel(np.asarray(d, float))
    return bool(np.all(np.abs(v - s) <= 0.5 * 10.0 ** (-d) * (1 + 1e-6)))


def simnao(c):
    return tuple("sim" if ci else "não" for ci in c)


NM = lambda P, x: nelder_mead_comp(P, x, 1e-10, 1e-12, 2000)   # interno dos exemplos
ok = True
print("Otimização — deck 4.3.2: barreira logarítmica")

# ------------------------------------------------------------ 1. Ideia (1D)
r = barrier_log(lambda x: x[0]**2, lambda x: x[0] - 1, [2.0], 1, 0.1, 0, 0, 3, NM)
xR = r.history[1:, 5]
print("\n1. min x^2 s.a. x - 1 >= 0, de x = 2: x_R = (1 + sqrt(1+2R))/2")
for Ri, xi in zip(r.history[1:, 1], xR):
    print("   R = %5g: x_R = %.4f  (exato %.4f)" % (Ri, xi, (1 + np.sqrt(1 + 2 * Ri)) / 2))
c = [confere(xR, [1.366, 1.048, 1.005], 3)]
print("  x_R = 1.366, 1.048, 1.005: %s" % simnao(c))
ok = ok and all(c)

# ------------------------------------------------------------ 2. O exemplo nas contas
f = lambda x: x[0]**2 + x[1]**2
g = lambda x: x[0] + x[1] - 1
xs = np.array([0.5, 0.5])
r = barrier_log(f, g, [1, 1], 1, 0.1, 0, 0, 4, NM)
H = r.history[1:]
Rv = H[:, 1]; X = H[:, 5:7]; u = H[:, 7]
gv = X.sum(axis=1) - 1; err = np.linalg.norm(X - xs, axis=1)
print("\n2. min x1^2 + x2^2 s.a. x1 + x2 - 1 >= 0, de (1,1), R^(0) = 1, c = 0.1 (x* = (1/2,1/2), u* = 1)")
print("  %6s %20s %10s %12s %12s %8s %6s %6s"
      % ("R", "x_R", "g(x_R)", "||x_R-x*||", "||x_R-x*||/R", "u_R", "P", "fora"))
for i in range(4):
    print("  %6g   (%.4f; %.4f) %10.4f %12.5f %12.4f %8.4f %6d %6d"
          % (Rv[i], X[i, 0], X[i, 1], gv[i], err[i], err[i] / Rv[i], u[i], H[i, 2], H[i, 3]))
print("  erro O(R): ||x_R - x*||/R -> 1/sqrt 2 = %.4f;  u_R = R/g = (1 + sqrt(1+4R))/2 -> 1;"
      "  u_R g(x_R) = R" % (1 / np.sqrt(2)))
c = [confere(X, np.repeat([0.809, 0.5458, 0.5050, 0.5005], 2).reshape(4, 2),
             np.repeat([3, 4, 4, 4], 2).reshape(4, 2)),
     confere(gv, [0.618, 0.0916, 0.0099, 0.0010], [3, 4, 4, 4]),
     confere(err, [0.437, 0.0648, 0.0070, 0.00071], [3, 4, 4, 5]),
     confere(u, [1.618, 1.092, 1.010, 1.001], 3),
     all(gv > 0)]
print("  x_R: %s | g(x_R): %s | ||x_R - x*||: %s | u_R: %s | todos admissíveis: %s" % simnao(c))
ok = ok and all(c)

# ------------------------------------------------------------ 3. Barreira inversa
R = 0.01
Pinv = lambda x: np.inf if g(x) <= 0 else f(x) + R / g(x)
ri = nelder_mead_comp(Pinv, [1, 1], 1e-10, 1e-12, 2000)
print("\n3. Barreira inversa P = f + R/g, R = 0.01, de (1,1): x_R = (%.4f; %.4f), ||x_R - x*|| = %.4f"
      % (ri.x[0], ri.x[1], np.linalg.norm(ri.x - xs)))
print("   log (tabela acima), R = 0.01: x_R = (%.4f; %.4f), ||x_R - x*|| = %.4f  -> inversa O(sqrt R), log O(R)"
      % (X[2, 0], X[2, 1], err[2]))
c = [confere(ri.x, [0.548, 0.548], 3), confere(X[2], [0.505, 0.505], 3)]
print("  inversa (0.548; 0.548): %s | log (0.505; 0.505): %s" % simnao(c))
ok = ok and all(c)

# ------------------------------------------------------------ 4. Restrição inativa
r = barrier_log(lambda x: (x[0] - 1)**2 + (x[1] - 1)**2, lambda x: 9 - x[0]**2 - x[1]**2,
                [0, 0], 1, 0.1, 0, 0, 2, NM)
H = r.history[1:]
print("\n4. Restrição inativa: min (x1-1)^2 + (x2-1)^2 s.a. 9 - x1^2 - x2^2 >= 0, de (0,0) (x* = (1,1), u* = 0)")
for row in H:
    print("   R = %4g: x_R = (%.4f; %.4f), g = %.4f, u_R = R/g = %.4f"
          % (row[1], row[5], row[6], 9 - row[5]**2 - row[6]**2, row[7]))
c = [confere(H[:, 5:7], [[0.882, 0.882], [0.986, 0.986]], 3), bool(H[1, 7] < H[0, 7] < 1)]
print("  x_R = (0.882; 0.882), (0.986; 0.986): %s | u_R a descer para 0: %s" % simnao(c))
ok = ok and all(c)

# ------------------------------------------------------------ 5. Chamada do slide
print("\n5. Chamada do slide: barrier_log(f, g, [1, 1], 1, 0.1, 1e-8, 1e-6, 12)")
print("   (interno por omissão: nelder_mead_comp; o slide usa o fminsearch do MATLAB)")
r = barrier_log(f, g, [1, 1], 1, 0.1, 1e-8, 1e-6, 12, verbose=True)
nfc = r.history[1:, 2]
print("   nit = %d (R = %.3g), u = %.4f, chamadas a P por ciclo %d-%d, nfev = chamadas a P + 1 = %d"
      % (r.nit, r.R, r.u[0], nfc.min(), nfc.max(), r.nfev))
print("   fora de X: %d; n_f efetivo = %d (nfev é majorante de n_f); %s"
      % (r.nfora, r.nfev_efetivo, r.message))
print("   nota: R no ciclo 9 = 0.1^8 calculado como %.17g > 1e-8 = tolcomp: J*R <= tolcomp"
      % r.history[9, 1])
print("         só pode valer a partir do ciclo 10 (R ~ 1e-9), qualquer que seja o minimizador interno")
print("   slide (fminsearch do MATLAB): nit = 10 (R = 1e-9), u ~ 1; nfev ~ 140-190 por ciclo")
c = [abs(r.u[0] - 1) <= 0.01]
print("  u ~ 1 (a menos de 1 %%): %s | nit = 10: %s (informativo: depende do minimizador interno)"
      % (simnao(c)[0], "sim" if r.nit == 10 else "não"))
ok = ok and all(c)

# ------------------------------------------------------------ 6. Comparação
print("\n6. Comparação exterior vs. interior: o mesmo Nelder–Mead interno (tolerâncias 1e-10),")
print("   arranque a quente; para no 1.º ciclo com ||x^(t) - x*|| <= 1e-4 (lido em history)")


def ate(H, tol=1e-4):
    """1.º ciclo t (linha de history) com ||x^(t) - x*|| <= tol."""
    e = np.linalg.norm(H[1:, 5:7] - xs, axis=1)
    return int(np.argmax(e <= tol)) + 1


rb = barrier_log(f, g, [1, 1], 1, 0.1, 0, 0, 12, NM)
tb = ate(rb.history); Hb = rb.history
nPb = int(Hb[tb, 4]); forab = int(Hb[1:tb + 1, 3].sum())
print("   barreira de (1,1), R^(0) = 1, c = 0.1:")
for row in Hb[1:tb + 1]:
    print("     t = %d  R = %8.1e  x = (%.5f, %.5f)  ||x - x*|| = %.1e  P = %3d (fora %2d)  acum. = %d"
          % (row[0], row[1], row[5], row[6], np.linalg.norm(row[5:7] - xs), row[2], row[3], row[4]))
print("     -> %d ciclos, %d chamadas a P, %d fora de X, n_f = %d (sem a avaliação final)"
      % (tb, nPb, forab, nPb - forab))
ext = {}
for x0 in ((0.0, 0.0), (1.0, 1.0)):
    re = penalty_exterior(f, g, lambda x: [], list(x0), 0.1, 10, 0, 0, 12, NM)
    te = ate(re.history)
    ext[x0] = (te, int(re.history[te, 3]), int(np.median(re.history[1:te + 1, 2])))
print("   exterior de (0,0), R^(0) = 0.1, c = 10: %d ciclos, n_f = %d" % ext[(0.0, 0.0)][:2])
print("   exterior de (1,1), R^(0) = 0.1, c = 10: %d ciclos, n_f = %d" % ext[(1.0, 1.0)][:2])
medb = int(np.median(Hb[1:tb + 1, 2]))
print("\n                                                 exterior  barreira")
print("   ciclos exteriores t                          %9d %9d" % (ext[(0.0, 0.0)][0], tb))
print("   por ciclo (mediana; n_f | chamadas a P)      %9d %9d" % (ext[(0.0, 0.0)][2], medb))
print("   total até ||x^(t) - x*|| <= 1e-4             %9d %9d" % (ext[(0.0, 0.0)][1], nPb))
c = [(tb, nPb, forab, nPb - forab) == (5, 743, 26, 717),
     ext[(0.0, 0.0)][:2] == (6, 844), ext[(1.0, 1.0)][:2] == (6, 864),
     (ext[(0.0, 0.0)][2], medb) == (140, 146)]
print("  barreira 5 ciclos, 743 P, 26 fora, n_f = 717: %s | exterior (0,0) 6 ciclos, 844: %s |"
      " exterior (1,1) 6 ciclos, 864: %s | medianas 140 e 146: %s" % simnao(c))
ok = ok and all(c)

# informativo: o mesmo com o nelder_mead do deck 3.2.3 (contração exterior aceite com <=)
from nelder_mead import nelder_mead  # noqa: E402  (deck 3.2.3)
rb3 = barrier_log(f, g, [1, 1], 1, 0.1, 0, 0, 12, lambda P, x: nelder_mead(P, x, 1e-10, 1e-12, 2000))
t3 = ate(rb3.history); H3 = rb3.history
print("   informativo — com nelder_mead do deck 3.2.3 (aceita a contração exterior com <=):")
print("   barreira de (1,1): %d ciclos, %d chamadas a P, %d fora de X, n_f = %d"
      % (t3, H3[t3, 4], H3[1:t3 + 1, 3].sum(), H3[t3, 4] - H3[1:t3 + 1, 3].sum()))

print("\nconfere com os slides: %s" % ("sim" if ok else "não"))


Otimização — deck 4.3.2: barreira logarítmica

1. min x^2 s.a. x - 1 >= 0, de x = 2: x_R = (1 + sqrt(1+2R))/2
   R =     1: x_R = 1.3660  (exato 1.3660)
   R =   0.1: x_R = 1.0477  (exato 1.0477)
   R =  0.01: x_R = 1.0050  (exato 1.0050)
  x_R = 1.366, 1.048, 1.005: sim

2. min x1^2 + x2^2 s.a. x1 + x2 - 1 >= 0, de (1,1), R^(0) = 1, c = 0.1 (x* = (1/2,1/2), u* = 1)
       R                  x_R     g(x_R)   ||x_R-x*|| ||x_R-x*||/R      u_R      P   fora
       1   (0.8090; 0.8090)     0.6180      0.43702       0.4370   1.6180    145      0
     0.1   (0.5458; 0.5458)     0.0916      0.06478       0.6478   1.0916    154      2
    0.01   (0.5050; 0.5050)     0.0099      0.00700       0.7002   1.0099    146      5
   0.001   (0.5005; 0.5005)     0.0010      0.00071       0.7064   1.0010    144      8
  erro O(R): ||x_R - x*||/R -> 1/sqrt 2 = 0.7071;  u_R = R/g = (1 + sqrt(1+4R))/2 -> 1;  u_R g(x_R) = R
  x_R: sim | g(x_R): sim | ||x_R - x*||: sim | u_R: sim | todos admissíveis: sim

3. 

  t         R  P(ciclo) fora(ciclo)  P(acum)         x1         x2         u1
  0         -         0           0        0   1.000000   1.000000          -
  1   1.0e+00       145           0      145   0.809017   0.809017   1.618034
  2   1.0e-01       154           2      299   0.545804   0.545804   1.091608
  3   1.0e-02       146           5      445   0.504951   0.504951   1.009902
  4   1.0e-03       144           8      589   0.500500   0.500499   1.000999
  5   1.0e-04       154          11      743   0.500050   0.500050   1.000100
  6   1.0e-05       208          26      951   0.500005   0.500005   1.000012
  7   1.0e-06       236          37     1187   0.500001   0.500000   0.999998
  8   1.0e-07       148          54     1335   0.500002   0.499998   1.000054
  9   1.0e-08       158          64     1493   0.500002   0.499998   0.999103
 10   1.0e-09       182          80     1675   0.500002   0.499998   1.007475
   nit = 10 (R = 1e-09), u = 1.0075, chamadas a P por ciclo 144-

   exterior de (0,0), R^(0) = 0.1, c = 10: 6 ciclos, n_f = 844
   exterior de (1,1), R^(0) = 0.1, c = 10: 6 ciclos, n_f = 864

                                                 exterior  barreira
   ciclos exteriores t                                  6         5
   por ciclo (mediana; n_f | chamadas a P)            140       146
   total até ||x^(t) - x*|| <= 1e-4                   844       743
  barreira 5 ciclos, 743 P, 26 fora, n_f = 717: sim | exterior (0,0) 6 ciclos, 844: sim | exterior (1,1) 6 ciclos, 864: sim | medianas 140 e 146: sim


   informativo — com nelder_mead do deck 3.2.3 (aceita a contração exterior com <=):
   barreira de (1,1): 5 ciclos, 739 chamadas a P, 26 fora de X, n_f = 713

confere com os slides: sim


## 4.3.3 Restrições na prática: penalizar, rejeitar ou comparar

Exemplo [`ex04_3_3_constraint_handling.py`](https://github.com/aguilarmadeira/optimization-course/blob/main/code/python/04_constrained_optimization/04_3_3_constraint_handling/ex04_3_3_constraint_handling.py) · funções da UC: [`hooke_jeeves.py`](https://github.com/aguilarmadeira/optimization-course/blob/main/code/python/03_multivariate_optimization/03_2_5_hooke_jeeves/hooke_jeeves.py), [`nelder_mead.py`](https://github.com/aguilarmadeira/optimization-course/blob/main/code/python/03_multivariate_optimization/03_2_3_nelder_mead/nelder_mead.py), [`extreme_barrier.py`](https://github.com/aguilarmadeira/optimization-course/blob/main/code/python/04_constrained_optimization/04_3_3_constraint_handling/extreme_barrier.py)

```text
Reproduz os números do deck 4.3.3 (restrições na prática).

Um só problema de referência, resolvido pelo MESMO Hooke–Jeeves da UC (3.2.5),
com a restrição comunicada ao algoritmo de cinco formas:

    min f(x) = (x1 - 2)^2 + (x2 - 1)^2   s.a.  g(x) = 2 - x1 - x2 >= 0,
    x* = (1.5, 0.5), f* = 0.5, u* = 1;  (2, 1) é inadmissível.
Convenção do capítulo 4: g >= 0, <g> = min(g, 0); violação v = -<g> = max(0, -g).

  A  ignorar a restrição:         f
  B  penalização quadrática:      f + R <g>^2 = f + R v^2,  R = 1, 10, 100, 1000
  C  penalização exata (L1):      f + R |<g>| = f + R v,  R = 0.5, 1, 2
  D  barreira extrema:            f se g >= 0, senão +inf (f não é avaliada)
  E  regra de admissibilidade:    compara (v, f), v = max(0, -g); nenhuma soma
     (hooke_jeeves com o argumento opcional menor)

Hooke–Jeeves: x0 = (0, 0), a = 2, P0 = 0.5, T = 1e-6 (todos os casos).
Também: (i) B analítico, v(x_R) = 1/(1 + 2R); (ii) restrição oculta: um
«simulador» que falha (NaN) sem dar g, tratado com +inf = D; (iii) E e D a
partir de x0 = (3, 3), inadmissível; (iv) o bloqueio em (1, 1) e a correção
com as direções rodadas (HJ nas variáveis y, x = Q y, Q = [1 1; 1 -1]/sqrt 2);
(v) informativo: o Nelder–Mead do 3.2.3 em C (R = 2) e D.
Funções: extreme_barrier.py (D e a restrição oculta); hooke_jeeves (3.2.5) e
nelder_mead (3.2.3), postas no caminho por uc_setup.
Os números dos slides saem daqui; os slides usam vírgula decimal.
```

In [6]:
import numpy as np

from hooke_jeeves import hooke_jeeves      # (3.2.5)
from nelder_mead import nelder_mead        # (3.2.3)
from extreme_barrier import extreme_barrier


def confere(v, s, d):
    """v confere com o valor s do slide, mostrado com d casas decimais?"""
    v = np.ravel(np.asarray(v, float)); s = np.ravel(np.asarray(s, float))
    d = np.ravel(np.asarray(d, float))
    return bool(np.all(np.abs(v - s) <= 0.5 * 10.0 ** (-d) * (1 + 1e-6)))


def simnao(c):
    return tuple("sim" if ci else "não" for ci in c)


# --- o problema (o mesmo em todo o deck) -----------------------------------
f = lambda x: (x[0] - 2)**2 + (x[1] - 1)**2
g = lambda x: 2 - x[0] - x[1]                      # g(x) >= 0 (convenção do cap. 4)
v = lambda x: max(0.0, -g(x))                      # violação v = -<g>
xs = np.array([1.5, 0.5])

# --- os cinco tratamentos: só muda a função (ou a comparação) --------------
FA = f                                                   # A: ignorar
FB = lambda R: (lambda x: f(x) + R * v(x)**2)            # B: quadrática
FC = lambda R: (lambda x: f(x) + R * v(x))               # C: exata (L1)
FD = lambda x: extreme_barrier(f, g, x)                  # D: barreira extrema
FE = lambda x: (v(x), f(x))                              # E: o par (v, f)
menorE = lambda a, b: a[0] < b[0] or (a[0] == b[0] and a[1] < b[1])   # E: regra

x0 = [0.0, 0.0]; a = 2; P0 = 0.5; T = 1e-6
HJ = lambda F, x0=x0, **k: hooke_jeeves(F, x0, a, P0, T, variante="1961", **k)

ok = True
print("Otimização — deck 4.3.3: restrições na prática (um problema, cinco tratamentos)")
print("min (x1-2)^2 + (x2-1)^2 s.a. g = 2 - x1 - x2 >= 0;  x* = (1.5, 0.5), f* = 0.5, u* = 1")
print("Hooke–Jeeves (3.2.5): x0 = (0, 0), a = 2, P0 = 0.5, T = 1e-6\n")


def linha(nome, r):
    x = r.x
    adm = "sim" if g(x) >= -1e-12 else "não"
    print("  %-26s x = (%.6f; %.6f)  f = %.6f  g = %+.2e  admissível: %-3s  n_f = %4d"
          % (nome, x[0], x[1], f(x), g(x), adm, r.nfev))
    return x


# ------------------------------------------------------------ 1. a tabela
print("1. A tabela final (slide «O mesmo problema, cinco tratamentos»)")
rA = HJ(FA); xA = linha("A ignorar", rA)
rB = {R: HJ(FB(R)) for R in (1, 10, 100, 1000)}
xB = {R: linha("B quadrática, R = %g" % R, rB[R]) for R in rB}
rC = {R: HJ(FC(R)) for R in (0.5, 1, 2)}
xC = {R: linha("C exata, R = %g" % R, rC[R]) for R in rC}
rD = HJ(FD); xD = linha("D barreira extrema", rD)
rE = HJ(FE, menor=menorE); xE = linha("E regra (v, f)", rE)

c = [confere(xA, [2, 1], 4),
     confere([xB[R] for R in (1, 10, 100, 1000)],
             [[1.667, 0.667], [1.524, 0.524], [1.502, 0.502], [1.500, 0.500]], 3),
     confere([v(xB[R]) for R in (1, 10, 100, 1000)], [0.33, 0.048, 0.0050, 0.00050], [2, 3, 4, 5]),
     confere([xC[R] for R in (0.5, 1, 2)], [[1.75, 0.75], [1.5, 0.5], [1, 1]], 4),
     confere([xD, xE], [[1, 1], [1, 1]], 4)]
print("  A = (2; 1) | B -> x* por fora, v = 0.33 ... 0.0005 | C: (1.75; 0.75), x*, (1; 1)"
      " | D = E = (1; 1): %s" % (simnao(c),))
ok = ok and all(c)
nfora = int(np.sum(np.isinf(rD.ftrace)))
c = [nfora == 45]
print("  D: n_F = %d chamadas, %d fora de X (f não avaliada): %s" % (rD.nfev, nfora, simnao(c)[0]))
ok = ok and all(c)
c = [np.array_equal(rD.history[:, 2:4], rE.history[:, 2:4]) and rD.nfev == rE.nfev]
print("  de x0 admissível, E passa pelos mesmos pontos base que D, com o mesmo n_F: %s" % simnao(c))
print("  (as tentativas em torno de pontos inadmissíveis diferem: D compara +inf com +inf, E compara v)")
ok = ok and all(c)

# ------------------------------------------------------------ 2. B analítico
print("\n2. B: x_R = (2, 1) - R v_R (1, 1), v_R = 1/(1 + 2R) (slide «Quanto deve valer R?»)")
print("  %6s %22s %10s %10s %12s %6s" % ("R", "x_R (HJ)", "v (HJ)", "1/(1+2R)", "||x_R-x*||", "n_f"))
for R in rB:
    x = xB[R]
    print("  %6g   (%.4f; %.4f) %10.5f %10.5f %12.5f %6d"
          % (R, x[0], x[1], v(x), 1 / (1 + 2 * R), np.linalg.norm(x - xs), rB[R].nfev))
c = [confere([v(xB[R]) for R in rB], [1 / (1 + 2 * R) for R in rB], 5)]
print("  HJ confere com 1/(1+2R) a 5 casas: %s" % simnao(c))
ok = ok and all(c)

# ------------------------------------------------------------ 3. restrição oculta
print("\n3. Restrição oculta: o «simulador» falha (NaN) e não diz porquê (não há g)")


def simulador(x):
    """Caixa negra: devolve um número, ou NaN quando «a simulação não converge»
    (aqui, quando x1 + x2 > 2), sem dizer porquê: quem a usa não conhece g."""
    return np.nan if x[0] + x[1] > 2 else (x[0] - 2)**2 + (x[1] - 1)**2


FH = lambda x: extreme_barrier(simulador, None, x)       # falhou -> +inf
rH = HJ(FH); xH = linha("oculta, +inf se falha", rH)
c = [np.array_equal(rH.history, rD.history, equal_nan=True)]
print("  a mesma corrida que D, sem nunca conhecer g: %s" % simnao(c))
ok = ok and all(c)

# ------------------------------------------------------------ 4. x0 inadmissível
print("\n4. De x0 = (3, 3), inadmissível (g = -4)")
rD3 = HJ(FD, [3.0, 3.0]); linha("D barreira extrema", rD3)
rE3 = HJ(FE, [3.0, 3.0], menor=menorE); xE3 = linha("E regra (v, f)", rE3)
c = [np.isinf(rD3.fx) and np.array_equal(rD3.x, [3, 3]), confere(xE3, [1, 1], 4)]
print("  D não sai de x0 (tudo +inf) | E chega à fronteira e para em (1; 1): %s" % (simnao(c),))
ok = ok and all(c)

# ------------------------------------------------------------ 5. o bloqueio em (1, 1)
print("\n5. Porque para em (1, 1)? As 4 tentativas +-P_j (P = 0.5) a partir de (1, 1):")
xb = np.array([1.0, 1.0]); P = 0.5
for d, nome in (([1, 0], "+e1"), ([-1, 0], "-e1"), ([0, 1], "+e2"), ([0, -1], "-e2")):
    xp = xb + P * np.array(d)
    print("  %s: x = (%.1f; %.1f)  f = %.2f  g = %+.1f  C(R=2) = %.2f  D = %s"
          % (nome, xp[0], xp[1], f(xp), g(xp), FC(2)(xp), "%.2f" % FD(xp) if g(xp) >= 0 else "+inf"))
print("  em (1, 1): f = %.2f; nenhuma direção coordenada melhora (qualquer P > 0);"
      " a direção (1, -1)/sqrt 2, ao longo da fronteira, melhora." % f(xb))
xq = xb + 0.25 * np.array([1, -1])                # (1.25, 0.75): na fronteira
c = [all(FC(2)(xb + p * np.array(d)) > f(xb) for p in (0.5, 1e-3, 1e-6)
         for d in ([1, 0], [-1, 0], [0, 1], [0, -1])),
     FD(xq) < f(xb)]
print("  nenhuma tentativa +-P e_j melhora em C (R = 2), P = 0.5, 1e-3, 1e-6 | (1,-1) melhora: %s"
      % (simnao(c),))
ok = ok and all(c)

# ------------------------------------------------------------ 6. a correção: rodar as direções
print("\n6. O mesmo Hooke–Jeeves nas variáveis y, x = Q y, Q = [1 1; 1 -1]/sqrt 2"
      " (direções ao longo e na normal à fronteira)")
Q = np.array([[1.0, 1.0], [1.0, -1.0]]) / np.sqrt(2)
rot = lambda F: (lambda y: F(Q @ y))
y0 = Q.T @ np.array(x0)
rotres = {}
for nome, F, k in (("C exata, R = 2", FC(2), {}), ("D barreira extrema", FD, {}),
                   ("E regra (v, f)", FE, {"menor": menorE})):
    r = hooke_jeeves(rot(F), y0, a, P0, T, variante="1961", **k)
    r.x = Q @ r.x
    rotres[nome] = linha(nome + " (rodado)", r)
c = [confere(list(rotres.values()), [xs] * 3, 4)]
print("  com as direções rodadas, C (R = 2), D e E chegam a x* = (1.5; 0.5): %s" % simnao(c))
ok = ok and all(c)

# ------------------------------------------------------------ 7. informativo: Nelder–Mead
print("\n7. Informativo: o Nelder–Mead do 3.2.3 (simplex inicial de x0 = (0, 0), tolx = 1e-8,"
      " tolf = 1e-10) chega a x* aqui, mas sem garantia geral")
for nome, F in (("C exata, R = 2", FC(2)), ("D barreira extrema", FD)):
    r = nelder_mead(F, np.array(x0), tolx=1e-8, tolf=1e-10, kmax=2000)
    linha(nome + " (Nelder–Mead)", r)

print("\nconfere com os slides: %s" % ("sim" if ok else "não"))


Otimização — deck 4.3.3: restrições na prática (um problema, cinco tratamentos)
min (x1-2)^2 + (x2-1)^2 s.a. g = 2 - x1 - x2 >= 0;  x* = (1.5, 0.5), f* = 0.5, u* = 1
Hooke–Jeeves (3.2.5): x0 = (0, 0), a = 2, P0 = 0.5, T = 1e-6

1. A tabela final (slide «O mesmo problema, cinco tratamentos»)
  A ignorar                  x = (2.000000; 1.000000)  f = 0.000000  g = -1.00e+00  admissível: não  n_f =   96
  B quadrática, R = 1        x = (1.666666; 0.666668)  f = 0.222222  g = -3.33e-01  admissível: não  n_f =  253
  B quadrática, R = 10       x = (1.523808; 0.523811)  f = 0.453515  g = -4.76e-02  admissível: não  n_f =  244
  B quadrática, R = 100      x = (1.502491; 0.502483)  f = 0.495038  g = -4.97e-03  admissível: não  n_f =  311
  B quadrática, R = 1000     x = (1.500256; 0.500244)  f = 0.499500  g = -5.00e-04  admissível: não  n_f =  644
  C exata, R = 0.5           x = (1.750000; 0.750000)  f = 0.125000  g = -5.00e-01  admissível: não  n_f =  100
  C exata, R = 1             x = (1.